# Preprocessing (Updated for `light_curves.csv`)
This notebook cleans ZTF-style light curve observations and produces:
- `light_curves_clean.csv`: cleaned observation table
- `light_curves_sequences.npz`: fixed-length per-object sequences for contrastive learning

**Input columns (expected):** `oid, mjd, fid, mag, e_mag, magpsf, sigmapsf, ra, dec, isdiffpos`

In [12]:
import os
import numpy as np
import pandas as pd

DATA_PATH = 'light_curves.csv'  # <-- new dataset
OUT_DIR = 'preprocessed'
os.makedirs(OUT_DIR, exist_ok=True)
# Ensure outputs are written into the workspace folder (absolute path)
WORKSPACE_ROOT = r'c:/Users/NIPUN/Desktop/LIGHTNING'
os.makedirs(os.path.join(WORKSPACE_ROOT, OUT_DIR), exist_ok=True)
clean_csv_path = os.path.join(WORKSPACE_ROOT, OUT_DIR, 'light_curves_clean.csv')
seq_npz_path = os.path.join(WORKSPACE_ROOT, OUT_DIR, 'light_curves_sequences.npz')

# Load
obs = pd.read_csv(DATA_PATH)
print('Loaded:', DATA_PATH)
print('Shape:', obs.shape)
print('Columns:', list(obs.columns))
obs.head()

Loaded: light_curves.csv
Shape: (20000, 10)
Columns: ['oid', 'mjd', 'fid', 'mag', 'e_mag', 'magpsf', 'sigmapsf', 'ra', 'dec', 'isdiffpos']


,oid,mjd,fid,mag,e_mag,magpsf,sigmapsf,ra,dec,isdiffpos
0,ZTF18aazeojq,58278.407130,1,NaN,NaN,16.692076,0.025775,307.792636,51.134943,-1
1,ZTF18aazeojq,58281.403681,1,NaN,NaN,16.631727,0.024336,307.792558,51.134826,-1
2,ZTF18aazeojq,58285.413102,2,NaN,NaN,16.383300,0.027792,307.792625,51.134461,1
3,ZTF18aazeojq,58287.404167,1,NaN,NaN,16.658768,0.025382,307.792841,51.134746,-1
4,ZTF18aazeojq,58288.407836,1,NaN,NaN,16.912527,0.146301,307.792625,51.135020,1


## 1) Standardize columns
We will use **PSF magnitude** for modeling:
- `mag_used` = `magpsf` (fallback to `mag` if needed)
- `err_used` = `sigmapsf` (fallback to `e_mag` if needed)

In [3]:
obs = obs.copy()

# normalize placeholders and infinities
obs.replace(['', ' ', 'NA', 'NaN', 'nan', 'None', 'none', 'NULL'], np.nan, inplace=True)
obs.replace([np.inf, -np.inf], np.nan, inplace=True)

# choose magnitude + error columns robustly
obs['mag_used'] = obs['magpsf']
obs.loc[obs['mag_used'].isna(), 'mag_used'] = obs.loc[obs['mag_used'].isna(), 'mag']

obs['err_used'] = obs['sigmapsf']
obs.loc[obs['err_used'].isna(), 'err_used'] = obs.loc[obs['err_used'].isna(), 'e_mag']

# keep only the columns we will use downstream
keep_cols = ['oid','mjd','fid','ra','dec','isdiffpos','mag_used','err_used']
missing = [c for c in keep_cols if c not in obs.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
obs = obs[keep_cols].copy()

print('After standardization:', obs.shape)
obs.head()

After standardization: (20000, 8)


,oid,mjd,fid,ra,dec,isdiffpos,mag_used,err_used
0,ZTF18aazeojq,58278.407130,1,307.792636,51.134943,-1,16.692076,0.025775
1,ZTF18aazeojq,58281.403681,1,307.792558,51.134826,-1,16.631727,0.024336
2,ZTF18aazeojq,58285.413102,2,307.792625,51.134461,1,16.383300,0.027792
3,ZTF18aazeojq,58287.404167,1,307.792841,51.134746,-1,16.658768,0.025382
4,ZTF18aazeojq,58288.407836,1,307.792625,51.135020,1,16.912527,0.146301


## 2) Basic cleaning
- drop duplicates
- drop rows missing critical fields (`oid, mjd, fid, mag_used`)
- enforce dtypes
- sanity filter RA/Dec ranges

In [4]:
# drop duplicates
before = len(obs)
obs = obs.drop_duplicates().reset_index(drop=True)
print('Dropped duplicates:', before - len(obs))

# drop rows missing critical fields
critical = ['oid','mjd','fid','mag_used']
before = len(obs)
obs = obs.dropna(subset=critical).reset_index(drop=True)
print('Dropped rows with missing critical fields:', before - len(obs))

# dtypes
obs['oid'] = obs['oid'].astype(str)
obs['mjd'] = pd.to_numeric(obs['mjd'], errors='coerce')
obs['fid'] = pd.to_numeric(obs['fid'], errors='coerce').astype('Int64')
obs['ra'] = pd.to_numeric(obs['ra'], errors='coerce')
obs['dec'] = pd.to_numeric(obs['dec'], errors='coerce')
obs['isdiffpos'] = pd.to_numeric(obs['isdiffpos'], errors='coerce').fillna(0).astype(int)
obs['mag_used'] = pd.to_numeric(obs['mag_used'], errors='coerce')
obs['err_used'] = pd.to_numeric(obs['err_used'], errors='coerce')

# sanity filter RA/Dec
if 'ra' in obs.columns:
    before = len(obs)
    obs = obs[(obs['ra'] >= 0) & (obs['ra'] <= 360)].reset_index(drop=True)
    print('Filtered RA outside [0,360]:', before - len(obs))
if 'dec' in obs.columns:
    before = len(obs)
    obs = obs[(obs['dec'] >= -90) & (obs['dec'] <= 90)].reset_index(drop=True)
    print('Filtered Dec outside [-90,90]:', before - len(obs))

# sort
obs = obs.sort_values(['oid','mjd','fid']).reset_index(drop=True)
print('Rows after cleaning:', len(obs))
obs.head()

Dropped duplicates: 264
Dropped rows with missing critical fields: 0
Filtered RA outside [0,360]: 0
Filtered Dec outside [-90,90]: 0
Rows after cleaning: 19736


,oid,mjd,fid,ra,dec,isdiffpos,mag_used,err_used
0,ZTF18aajtltx,58252.436134,1,274.382237,58.493638,-1,19.838300,0.104995
1,ZTF18aajtltx,58252.443218,1,274.382248,58.493561,-1,19.928700,0.133612
2,ZTF18aajtltx,58252.491157,2,274.382250,58.493683,-1,19.125100,0.137872
3,ZTF18aajtltx,58261.453738,1,274.382241,58.493638,-1,20.198800,0.143248
4,ZTF18aajtltx,58272.391377,1,274.382412,58.493586,-1,19.766142,0.178495


## 3) Outlier clipping (CL-friendly)
We clip extreme values for `mag_used` and `err_used` (not RA/Dec) to robust percentiles to avoid breaking rare patterns.

In [5]:
def clip_series(s: pd.Series, lo_q=0.01, hi_q=0.99) -> pd.Series:
    lo = s.quantile(lo_q)
    hi = s.quantile(hi_q)
    if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
        return s.clip(lo, hi)
    return s

for col in ['mag_used','err_used']:
    if col in obs.columns:
        before_min, before_max = obs[col].min(), obs[col].max()
        obs[col] = clip_series(obs[col])
        after_min, after_max = obs[col].min(), obs[col].max()
        print(col, 'min/max:', (before_min, before_max), '->', (after_min, after_max))

mag_used min/max: (np.float64(13.544434), np.float64(20.5377)) -> (np.float64(14.91790455), np.float64(19.37839))
err_used min/max: (np.float64(0.009013123), np.float64(0.499309)) -> (np.float64(0.011751), np.float64(0.2305998500000009))


## 4) Save cleaned observation table

In [13]:
obs.to_csv(clean_csv_path, index=False)
print('Saved cleaned CSV ->', clean_csv_path)
obs.head()

Saved cleaned CSV -> c:/Users/NIPUN/Desktop/LIGHTNING\preprocessed\light_curves_clean.csv


,oid,mjd,fid,mag,e_mag,magpsf,sigmapsf,ra,dec,isdiffpos
0,ZTF18aazeojq,58278.407130,1,NaN,NaN,16.692076,0.025775,307.792636,51.134943,-1
1,ZTF18aazeojq,58281.403681,1,NaN,NaN,16.631727,0.024336,307.792558,51.134826,-1
2,ZTF18aazeojq,58285.413102,2,NaN,NaN,16.383300,0.027792,307.792625,51.134461,1
3,ZTF18aazeojq,58287.404167,1,NaN,NaN,16.658768,0.025382,307.792841,51.134746,-1
4,ZTF18aazeojq,58288.407836,1,NaN,NaN,16.912527,0.146301,307.792625,51.135020,1


## 5) Build fixed-length per-object sequences
For contrastive learning, we often want a **consistent length** per object.

We will:
1. group by `oid`
2. normalize time per object: `t = (mjd - min_mjd)`
3. create a uniform time grid of length `L`
4. for each `fid` band, interpolate `mag_used` and `err_used` onto the grid
5. produce arrays:
   - `X_mag`: shape `[N, L, B]`
   - `X_err`: shape `[N, L, B]`
   - `X_mask`: shape `[N, L, B]` (1 where real observations existed nearby)
   - `t_grid`: shape `[L]`

This is a strong starting format for SimCLR/BYOL augmentations.

In [7]:
from collections import defaultdict

L = 128  # sequence length (you can change)
min_points_per_oid = 5  # keep objects with at least this many obs

# Determine bands present
bands = sorted([int(b) for b in obs['fid'].dropna().unique()])
B = len(bands)
print('Bands (fid):', bands)

# Filter objects with enough observations
counts = obs.groupby('oid').size()
keep_oids = counts[counts >= min_points_per_oid].index
obs_f = obs[obs['oid'].isin(keep_oids)].copy()
print('Objects kept:', len(keep_oids), '/', counts.shape[0])

def interp_with_mask(t_src: np.ndarray, y_src: np.ndarray, t_grid: np.ndarray):
    """Linear interpolation + a simple proximity mask."""
    t_src = np.asarray(t_src, dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)

    m = np.isfinite(t_src) & np.isfinite(y_src)
    t_src = t_src[m]
    y_src = y_src[m]

    if len(t_src) < 2:
        return (np.full_like(t_grid, np.nan, dtype=np.float32),
                np.zeros_like(t_grid, dtype=np.float32))

    order = np.argsort(t_src)
    t_src = t_src[order]
    y_src = y_src[order]

    y = np.interp(t_grid, t_src, y_src).astype(np.float32)

    # proximity mask: 1 if there is a real obs within a small window of the grid time
    span = float(t_src.max() - t_src.min())
    win = max(0.1, 0.01 * (span + 1e-6))  # days
    d = np.min(np.abs(t_grid[:, None] - t_src[None, :]), axis=1)
    mask = (d <= win).astype(np.float32)
    return y, mask

# Build sequences
oid_list = []
X_mag = []
X_err = []
X_mask = []

for oid, g in obs_f.groupby('oid'):
    g = g.sort_values('mjd')
    t0 = float(g['mjd'].min())
    t = (g['mjd'].astype(float) - t0).to_numpy()

    # If the object has no time span (all same mjd), skip
    t_span = float(np.nanmax(t) - np.nanmin(t))
    if not np.isfinite(t_span) or t_span <= 0:
        continue

    t_grid = np.linspace(0.0, t_span, L).astype(np.float32)

    mag_cube = np.full((L, B), np.nan, dtype=np.float32)
    err_cube = np.full((L, B), np.nan, dtype=np.float32)
    mask_cube = np.zeros((L, B), dtype=np.float32)

    for j, fid in enumerate(bands):
        gb = g[g['fid'] == fid]
        if gb.empty:
            continue

        y_mag, m_mag = interp_with_mask(
            t_src=(gb['mjd'].astype(float) - t0).to_numpy(),
            y_src=gb['mag_used'].astype(float).to_numpy(),
            t_grid=t_grid.astype(np.float64),
        )
        y_err, _ = interp_with_mask(
            t_src=(gb['mjd'].astype(float) - t0).to_numpy(),
            y_src=gb['err_used'].astype(float).to_numpy(),
            t_grid=t_grid.astype(np.float64),
        )

        mag_cube[:, j] = y_mag
        err_cube[:, j] = y_err
        mask_cube[:, j] = m_mag

    # Require at least some coverage
    if mask_cube.sum() < 3:
        continue

    oid_list.append(oid)
    X_mag.append(mag_cube)
    X_err.append(err_cube)
    X_mask.append(mask_cube)

X_mag = np.stack(X_mag, axis=0) if len(X_mag) else np.zeros((0, L, B), dtype=np.float32)
X_err = np.stack(X_err, axis=0) if len(X_err) else np.zeros((0, L, B), dtype=np.float32)
X_mask = np.stack(X_mask, axis=0) if len(X_mask) else np.zeros((0, L, B), dtype=np.float32)

print('Sequence tensors:')
print(' - X_mag:', X_mag.shape)
print(' - X_err:', X_err.shape)
print(' - X_mask:', X_mask.shape)

# Normalize magnitudes per object (median-center per band)
X_mag_norm = X_mag.copy()
for j in range(B):
    band = X_mag_norm[:, :, j]
    med = np.nanmedian(band, axis=1, keepdims=True)
    X_mag_norm[:, :, j] = band - med

# Save
np.savez_compressed(
    seq_npz_path,
    oid=np.array(oid_list),
    bands=np.array(bands, dtype=np.int32),
    L=np.int32(L),
    X_mag=X_mag_norm,
    X_err=X_err,
    X_mask=X_mask,
)
print('Saved sequences NPZ ->', seq_npz_path)
seq_npz_path


Bands (fid): [1, 2]
Objects kept: 5 / 5
Sequence tensors:
 - X_mag: (5, 128, 2)
 - X_err: (5, 128, 2)
 - X_mask: (5, 128, 2)
Saved sequences NPZ -> /preprocessed\light_curves_sequences.npz


'/preprocessed\\light_curves_sequences.npz'

## 6) Quick sanity checks

In [8]:
print('Number of objects:', len(oid_list))
print('Fraction of NaNs in X_mag:', np.isnan(X_mag_norm).mean())
print('Mean mask coverage:', np.mean(X_mask))

# show one example object summary
idx = 0
print('Example oid:', oid_list[idx])
print('Non-empty points per band (mask sum):', X_mask[idx].sum(axis=0))

Number of objects: 5
Fraction of NaNs in X_mag: 0.0
Mean mask coverage: 0.8890625
Example oid: ZTF18aajtltx
Non-empty points per band (mask sum): [118. 122.]
